# Agglomerative (Hierarchical) Clustering

## What is Hierarchical Clustering?
Hierarchical Clustering builds a **tree of clusters** (called a dendrogram) by iteratively merging the two closest clusters.

### Two Strategies
| Strategy | Direction | How it works |
|----------|-----------|-------------|
| **Agglomerative** (bottom-up) | ↑ merge | Start with each point as its own cluster; merge closest pairs until one cluster |
| **Divisive** (top-down) | ↓ split | Start with one cluster; split repeatedly |

**sklearn implements Agglomerative** — the more common approach.

### Linkage Criteria (how "distance between clusters" is defined)
| Linkage | Distance between clusters A and B |
|---------|-----------------------------------|
| **Ward** | Increase in total within-cluster variance after merging (minimises WCSS) |
| **Complete** | max distance between any point in A and any point in B |
| **Average** | mean distance between all pairs (one from A, one from B) |
| **Single** | min distance between any point in A and any point in B |

**Ward linkage** (used here) tends to produce the most compact, evenly-sized clusters.

### Key Advantage Over K-Means
- No need to specify K upfront — read it from the dendrogram
- Can capture non-spherical cluster shapes
- Deterministic — no random initialisation

### This Notebook
1. Load mall customer data
2. Plot a **dendrogram** to visually choose the number of clusters
3. Fit AgglomerativeClustering with K=5
4. Visualise the clusters


## Step 1: Imports

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
%matplotlib inline

## Step 2: Load the Dataset
**Mall Customer Segmentation dataset** — 200 customers with:
- CustomerID, Gender, Age, Annual Income (k$), Spending Score (1-100)

We use only **Annual Income** and **Spending Score** (columns 3 and 4) for 2D clustering.


In [ ]:
customer_data = pd.read_csv(r'd:\PENDRIVE 32 GB\Game\AI_PROJECTS\Machine Learning\100-days-of-machine-learning-main\agglomerative-clustering\hierarchical-clustering-with-python-and-scikit-learn-shopping-data.csv')

### Dataset Shape

In [ ]:
customer_data.shape

### Preview First 5 Rows

In [ ]:
customer_data.head()

## Step 3: Extract Features
Use only columns 3–4: **Annual Income (k$)** and **Spending Score (1–100)**.
These two features make the clearest clusters in this dataset.


In [ ]:
data = customer_data.iloc[:, 3:5].values
print(f'Feature matrix shape: {data.shape}')
data[:5]

## Step 4: Dendrogram — Choose the Number of Clusters

A **dendrogram** is a tree diagram that shows the merging history:
- Each leaf = one data point
- Each merge = one horizontal line at height = distance between merged clusters
- **Reading the dendrogram:** draw a horizontal line where there's the largest vertical gap — the number of vertical lines it crosses = optimal K

In this dataset, the largest gap is at K=5 (5 vertical lines crossed).

**`method='ward'`** = Ward linkage (minimises within-cluster variance at each merge step)


In [ ]:
import scipy.cluster.hierarchy as shc

plt.figure(figsize=(12, 7))
plt.title("Customer Dendrogram — Ward Linkage")
plt.xlabel("Customer Index")
plt.ylabel("Distance (Ward)")
dend = shc.dendrogram(shc.linkage(data, method='ward'))
plt.tight_layout()
plt.show()

## Step 5: Fit AgglomerativeClustering (K=5)
From the dendrogram we chose **K=5**.

**Parameters:**
- `n_clusters=5` — number of clusters (read from dendrogram)
- `metric='euclidean'` — straight-line distance between points
- `linkage='ward'` — merge strategy (minimise variance increase)

> **Note:** In sklearn ≥ 1.2 the `affinity` parameter was renamed to `metric`.


In [ ]:
from sklearn.cluster import AgglomerativeClustering

cluster = AgglomerativeClustering(n_clusters=5, metric='euclidean', linkage='ward')
labels_ = cluster.fit_predict(data)
print(f"Cluster labels (first 20): {labels_[:20]}")

### Cluster Label Array
One label per customer — values 0–4.

In [ ]:
labels_

## Step 6: Visualise the Clusters
Each colour = one of the 5 customer segments.
The `rainbow` colormap maps integer labels (0–4) to distinct colours.

**Expected segments in mall data:**
- High income, high spending (target customers)
- High income, low spending (potential upsell)
- Average income, average spending (bulk of customers)
- Low income, high spending (risky/impulse buyers)
- Low income, low spending (budget-conscious)


In [ ]:
plt.figure(figsize=(10, 7))
scatter = plt.scatter(data[:, 0], data[:, 1], c=cluster.labels_, cmap='rainbow', edgecolors='k', linewidths=0.3)
plt.colorbar(scatter, label='Cluster')
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.title('Agglomerative Clustering — Mall Customers (K=5)')
plt.tight_layout()
plt.show()

## Step 7: Cluster Sizes

In [ ]:
unique, counts = np.unique(labels_, return_counts=True)
for c, n in zip(unique, counts):
    print(f"Cluster {c}: {n} customers ({n/len(labels_)*100:.1f}%)")

## Summary

```
Agglomerative Clustering — Key Points:

  Algorithm:  Bottom-up hierarchical merging
  Linkage:    Ward (minimises within-cluster variance)
  Dataset:    200 mall customers, 2 features
  K chosen:   5 (read from the dendrogram's largest gap)

  Steps:
    1. Each point starts as its own cluster (200 clusters)
    2. Merge two closest clusters (by Ward distance)
    3. Repeat until K clusters remain

  Pros:
    ✓ No random initialisation → deterministic result
    ✓ Dendrogram shows natural K visually
    ✓ Works with any distance metric

  Cons:
    ✗ O(n² log n) time — slow for large datasets
    ✗ Cannot undo a merge once made

  sklearn:  AgglomerativeClustering(n_clusters=K, metric='euclidean', linkage='ward')
```
